<b><font size="6"> Data Mining Project - Group 40 2025/2026 </font></b><br><br>
<i> <font size="4"> Amazing International Airlines Inc. (AIAI)</font></i>

`Group 40`

20250380 Maria Fonseca <br>
20250405 Ana Macedo <br>
20250453 Lourenço Silva <br>

### <font color='#BFD72F'>**Methodology** </font> <a class="anchor" id='toc'></a> 


- [1. Import libraries](#1) 
- [2. Loading the Data](#2) 
- [3. Metadata](#3)
- [4. Business Understanding](#4)
- [5. Data Preprocessing](#6)
    - [5.1. Duplicates](#6_1)
    - [5.2. Incoherences](#6_2)
    - [5.3. Missing Values](#6_3)
    - [5.4. Outliers](#6_4)

- [7. Feature Engineering](#7)
    - [7.1 FlightsDB](#7_1)
        - [7.1.1 Selecting Created Features](#7_1_1)
    - [7.2 CustomerDB](#7_2)
        - [7.2.1 Selecting Created Features](#7_2_1)
    - [7.3 Feature Engineering with Merged Datasets](#7_3)
        - [7.3.1 Merge Datasets](#7_3_1)
        - [7.3.2 Add Features](#7_3_2)
        - [7.3.3 Select Created Features](#7_3_3)
    - [7.4 Summary of Engineered Features](#7_4)





<a class="anchor" id="1">

# **1. Import libraries**

[Back to TOC](#toc)
</a>

In [55]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

# To show all columns in the dataframe
!pip install tabulate
import itertools
from itertools import zip_longest
from tabulate import tabulate

# for better resolution plots
%config InlineBackend.figure_format = 'retina'

np.random.seed(40311)
sns.set()

<a class="anchor" id="2">

# **2. Loading the Data**

[Back to TOC](#toc)
</a>

In [56]:
flightsDB = pd.read_csv('../../data/DM_AIAI_FlightsDB.csv', sep = ",")
customerDB = pd.read_csv('../../data/DM_AIAI_CustomerDB.csv', sep = ",")
metaData = pd.read_csv('../../data/DM_AIAI_Metadata.csv', sep = ";", header= None)

Remove the 'Unnamed' column referring to a sequential numbering of the rows, as we set the column "Loyalty#" as the index

In [57]:
customerDB = customerDB.iloc[:, 1:]
customerDB

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
0,480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,female,Bachelor,Urban,70146.0,Married,Star,2/15/2019,NaN,3839.14,Standard
1,549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,T3G 6Y6,male,College,Rural,0.0,Divorced,Star,3/9/2019,NaN,3839.61,Standard
2,429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,male,College,Urban,0.0,Single,Star,7/14/2017,1/8/2021,3839.75,Standard
3,608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,2/17/2016,NaN,3839.75,Standard
4,530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,J8Y 3Z5,male,Bachelor,Suburban,97832.0,Married,Star,10/25/2017,NaN,3842.79,2021 Promotion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16916,100012,Ethan,Thompson,Ethan Thompson,Canada,Quebec,Quebec City,46.759733,-71.141009,Y0C 7D6,male,Bachelor,Suburban,NaN,Single,Star,2/27/2019,2/27/2019,NaN,Standard
16917,100013,Layla,Young,Layla Young,Canada,Alberta,Edmonton,53.524829,-113.546357,L3S 9Y3,female,Bachelor,Rural,NaN,Married,Star,9/20/2017,9/20/2017,NaN,Standard
16918,100014,Amelia,Bennett,Amelia Bennett,Canada,New Brunswick,Moncton,46.051866,-64.825428,G2S 2B6,male,Bachelor,Rural,NaN,Married,Star,11/28/2020,11/28/2020,NaN,Standard
16919,100015,Benjamin,Wilson,Benjamin Wilson,Canada,Quebec,Quebec City,46.862970,-71.133444,B1Z 8T3,female,College,Urban,NaN,Married,Star,4/9/2020,4/9/2020,NaN,Standard


<a class="anchor" id="3">

# **3. Metadata**

[Back to TOC](#toc)
</a>

**FlightsDB Database feature Description**
- **Loyalty#:**	Unique customer identifier linking to CustomerDB
- **Year:**	Year of flight activity record
- **Month:**	Month of flight activity record (1-12)
- **YearMonthDate:**	First day of the month for the activity period
- **NumFlights:**	Total number of flights taken by customer in the month
- **NumFlightsWithCompanions:**	Number of flights where customer traveled with companions
- **DistanceKM:**	Total distance traveled in kilometers for the month
- **PointsAccumulated:**	Loyalty points earned by customer during the month
- **PointsRedeemed:**	Loyalty points spent/redeemed by customer during the month
- **DollarCostPointsRedeemed:**	Dollar value of points redeemed during the month

**CustomerDB Database feature Description**
- **Loyalty#:**  Unique customer identifier for loyalty program members
- **First Name:**   Customer's first name
- **Last Name:**   Customer's last name 
- **Customer Name:** Customer's full name (concatenated)
- **Country:**	Customer's country of residence
- **Province or State:**	Customer's province or state
- **City:**	Customer's city of residence
- **Latitude:**	Geographic latitude coordinate of customer location
- **Longitude:**	Geographic longitude coordinate of customer locatio
- **Postal code:**	Customer's postal/ZIP code
- **Gender:**	Customer's gender
- **Education:**	Customer's highest education level (Bachelor, College, etc.)
- **Location:** Code	Urban/Suburban/Rural classification of customer residence
- **Income:**	Customer's annual income
- **Marital Status:**	Customer's marital status (Married, Single, Divorced)
- **LoyaltyStatus:**	Current tier status in loyalty program (Star > Nova > Aurora)
- **EnrollmentDateOpening:**	Date when customer joined the loyalty program
- **CancellationDate:**	Date when customer left the program
- **Customer Lifetime:** Value	Total calculated monetary value of customer relationship
- **EnrollmentType:**	Method of joining loyalty program

<a class="anchor" id="4">

# **4. Business Understanding**

[Back to TOC](#toc)
</a>

Amazing International Airlines Inc. (AIAI) operates in a highly competitive industry, where customer retention and loyalty are essential for profitability. 

The company has a loyalty program but lacks personalization and individualized engagement strategies, limiting its ability to achieve high levels of customer satisfaction. As a result, marketing campaigns and rewards are often generic, reducing the effectiveness of loyalty initiatives. 

The company’s main challenge is to reduce customer churn and optimize the performance of its loyalty program. Without detailed insights into the different customer profiles, it becomes difficult to offer personalized services and targeted offers that enhance satisfaction and engagement. 

The main goal is to implement a data-driven approach to segment AIAI’s customers, identifying distinct groups based on behavioral, demographic, and value-related factors. This segmentation will allow the company to create personalized marketing strategies, develop targeted loyalty offers, and ultimately reduce customer churn by 5%, which corresponds to approximately €50,000 in monthly savings. To support these business goals, the project will aim to develop a clustering model capable of distinguishing customer profiles, create customer segments and ensure the model is scalable and adaptable to evolving customer behavior.

 The project will follow five main stages, aligned with the CRISP-DM methodology. 
 It will begin with an exploratory data analysis (EDA) to understand the available data, assess its quality, and identify potential issues such as missing values or outliers. 
 The second stage will involve data preprocessing, including cleaning, normalization, and transformation to prepare the dataset for clustering. 
 Next, clustering techniques will be applied to identify distinct customer groups based on behavioral, demographic, and value-related factors. 
 The fourth stage will focus on evaluating the model’s performance and interpreting the resulting segments to extract actionable marketing insights. Finally, the model will be deployed and continuously monitored to ensure scalability and adaptability as customer behavior evolves.

<a class="anchor" id="5">

# **5. Data Preprocessing**

[Back to TOC](#toc)
</a>

<a class="anchor" id="5_1">

## **5.1.** Duplicates

[Back to TOC](#toc)
</a>

In [58]:
flightsDB[flightsDB.duplicated(keep=False)]
#This code will display the duplicated rows in the flight dataset which are 5778 rows in Total

,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
450,727091,2021,12,12/1/2021,0.0,0.0,0.0,0.0,0.0,0.0
535,369638,2021,12,12/1/2021,0.0,0.0,0.0,0.0,0.0,0.0
762,750578,2020,6,6/1/2020,0.0,0.0,0.0,0.0,0.0,0.0
941,547522,2020,6,6/1/2020,0.0,0.0,0.0,0.0,0.0,0.0
952,819842,2020,6,6/1/2020,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
607696,960050,2019,12,12/1/2019,0.0,0.0,0.0,0.0,0.0,0.0
608110,981508,2019,12,12/1/2019,0.0,0.0,0.0,0.0,0.0,0.0
608111,981508,2019,12,12/1/2019,0.0,0.0,0.0,0.0,0.0,0.0
608263,990512,2019,12,12/1/2019,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
customerDB[customerDB.duplicated(keep=False)]
#This code will display the duplicated rows in the customer dataset which are 0 rows in Total

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType


In [60]:
flightsDB.drop_duplicates(inplace=True)
customerDB.drop_duplicates(inplace=True)

<a class="anchor" id="5_2">

## **5.2.** Incoherences

[Back to TOC](#toc)
</a>

<a class="anchor" id="5_2_1">

## **5.2.1.** Number of Flights as floats

[Back to TOC](#toc)
</a>

In [61]:
flightsDB["NumFlights"] = flightsDB["NumFlights"].astype(int)
flightsDB["NumFlightsWithCompanions"] = flightsDB["NumFlightsWithCompanions"].astype(int)

In [62]:
total_accumulated = flightsDB.groupby('Loyalty#')['PointsAccumulated'].sum()
#val1[:-1]

total_redeemed = flightsDB.groupby('Loyalty#')['PointsRedeemed'].sum()
#val2[:-1]

#inconsitency in 458 observations
(total_redeemed > total_accumulated).sum()

np.int64(458)

<a class="anchor" id="5_3">

## **5.3.** Missing Values

[Back to TOC](#toc)
</a>

From what we've seen before we have:
- 0 missing values in the flights data set

- 14661 missing values in the customer data set. 
    - Income = 20 (~0.12%)	
    - CancellationDate = 14611 (~86%)	
    - Customer Lifetime Value = 20 (~0.12% )

In [63]:
customerDB

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
0,480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,female,Bachelor,Urban,70146.0,Married,Star,2/15/2019,NaN,3839.14,Standard
1,549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,T3G 6Y6,male,College,Rural,0.0,Divorced,Star,3/9/2019,NaN,3839.61,Standard
2,429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,male,College,Urban,0.0,Single,Star,7/14/2017,1/8/2021,3839.75,Standard
3,608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,2/17/2016,NaN,3839.75,Standard
4,530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,J8Y 3Z5,male,Bachelor,Suburban,97832.0,Married,Star,10/25/2017,NaN,3842.79,2021 Promotion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16916,100012,Ethan,Thompson,Ethan Thompson,Canada,Quebec,Quebec City,46.759733,-71.141009,Y0C 7D6,male,Bachelor,Suburban,NaN,Single,Star,2/27/2019,2/27/2019,NaN,Standard
16917,100013,Layla,Young,Layla Young,Canada,Alberta,Edmonton,53.524829,-113.546357,L3S 9Y3,female,Bachelor,Rural,NaN,Married,Star,9/20/2017,9/20/2017,NaN,Standard
16918,100014,Amelia,Bennett,Amelia Bennett,Canada,New Brunswick,Moncton,46.051866,-64.825428,G2S 2B6,male,Bachelor,Rural,NaN,Married,Star,11/28/2020,11/28/2020,NaN,Standard
16919,100015,Benjamin,Wilson,Benjamin Wilson,Canada,Quebec,Quebec City,46.862970,-71.133444,B1Z 8T3,female,College,Urban,NaN,Married,Star,4/9/2020,4/9/2020,NaN,Standard


In [64]:
# fill missing values in Income with median
median_income = customerDB["Income"].median()
customerDB["Income"].fillna(median_income, inplace=True)

# fill missing values in CLV with median
median_clv =customerDB["Customer Lifetime Value"].median()
customerDB["Customer Lifetime Value"].fillna(median_clv, inplace=True)

# fill missing values in CancellationDate with "Active" - it means that a customer is still loyal to the company
customerDB["CancellationDate"] = customerDB["CancellationDate"].fillna("Active")

/var/folders/s9/nfpy2jls4ps2k_p8lpjz6_yr0000gn/T/ipykernel_70178/2882134367.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  customerDB["Income"].fillna(median_income, inplace=True)
/var/folders/s9/nfpy2jls4ps2k_p8lpjz6_yr0000gn/T/ipykernel_70178/2882134367.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting val

In [65]:
customerDB.isna().sum().sum()

np.int64(0)

<a class="anchor" id="5_4">

## **5.4.** Outliers

[Back to TOC](#toc)
</a>

In [66]:
from boxplot_function import create_boxplots
numerical_columns = flightsDB.select_dtypes(include=['int64', 'float64']).columns.tolist()


In [67]:
#create_boxplots(flightsDB, numerical_columns)

<a class="anchor" id="5_4_1">

## **5.4.1.** Outliers Flights Dataset

[Back to TOC](#toc)
</a>

In [68]:
# Create new binary columns indicating if points or dollars were redeemed
flightsDB["did_redeemed"] = np.where(flightsDB["PointsRedeemed"] > 0, 1, 0)
flightsDB["value_redeemed"] = np.where(flightsDB["DollarCostPointsRedeemed"] > 0, 1, 0)

# Cap extreme values with the maximum value 
max_distance = flightsDB["DistanceKM"].max()
flightsDB["DistanceKM"] = np.where(flightsDB["DistanceKM"] > max_distance, max_distance, flightsDB["DistanceKM"])

max_points_accumulated = flightsDB["PointsAccumulated"].max()
flightsDB["PointsAccumulated"] = np.where(flightsDB["PointsAccumulated"] > max_points_accumulated, max_points_accumulated, flightsDB["PointsAccumulated"])

<a class="anchor" id="5_4_2">

## **5.4.2.** Outliers Customer Dataset

[Back to TOC](#toc)
</a>

In [76]:
# Create new feature with log transformation of CLV
customerDB["log_CLV"] = np.log1p(customerDB["Customer Lifetime Value"]) 

<a class="anchor" id="7">

# **7. Feature Engineering**

[Back to TOC](#toc)
</a>

Now, we will re-import the data using loyalty as the index again, since it becomes relevant for the feature engineering stage.
At that point, having loyalty as the index will help organize and reference the data more effectively.

In [71]:
flightsDB = pd.read_csv('../../data/DM_AIAI_FlightsDB.csv', sep = ",", index_col= "Loyalty#")
customerDB = pd.read_csv('../../data/DM_AIAI_CustomerDB.csv', sep = ",", index_col= "Loyalty#")

Remove the 'Unnamed' column referring to a sequential numbering of the rows, as we set the column "Loyalty#" as the index

In [72]:
customerDB = customerDB.iloc[:, 1:]
customerDB

,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
Loyalty#,,,,,,,,,,,,,,,,,,,
480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,female,Bachelor,Urban,70146.0,Married,Star,2/15/2019,NaN,3839.14,Standard
549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,T3G 6Y6,male,College,Rural,0.0,Divorced,Star,3/9/2019,NaN,3839.61,Standard
429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,male,College,Urban,0.0,Single,Star,7/14/2017,1/8/2021,3839.75,Standard
608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,2/17/2016,NaN,3839.75,Standard
530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,J8Y 3Z5,male,Bachelor,Suburban,97832.0,Married,Star,10/25/2017,NaN,3842.79,2021 Promotion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100012,Ethan,Thompson,Ethan Thompson,Canada,Quebec,Quebec City,46.759733,-71.141009,Y0C 7D6,male,Bachelor,Suburban,NaN,Single,Star,2/27/2019,2/27/2019,NaN,Standard
100013,Layla,Young,Layla Young,Canada,Alberta,Edmonton,53.524829,-113.546357,L3S 9Y3,female,Bachelor,Rural,NaN,Married,Star,9/20/2017,9/20/2017,NaN,Standard
100014,Amelia,Bennett,Amelia Bennett,Canada,New Brunswick,Moncton,46.051866,-64.825428,G2S 2B6,male,Bachelor,Rural,NaN,Married,Star,11/28/2020,11/28/2020,NaN,Standard


<a class="anchor" id="7_1">

## **7.1.** FlightsDB

[Back to TOC](#toc)
</a>

<a class="anchor" id="7_1_1">

### **7.1.1** Points Redemption Ratio 
[Back to TOC](#toc)
</a>

In [73]:
# PointsRedemptionRatio (Total)
flightsDB_new['PointsRedemptionRatio'] = np.where(
    flightsDB_new['TotalPointsAccumulated'] > 0,
    flightsDB_new['TotalPointsRedeemed'] / flightsDB_new['TotalPointsAccumulated'],
    0
)

NameError: name 'flightsDB_new' is not defined

<a class="anchor" id="7_1_2">

### **7.1.2** Total Flight With Companion Ratio
[Back to TOC](#toc)
</a>

In [ ]:
# Total Flight With Companion Ratio (lifetime)
flightsDB_new['TotalFlightWithCompanionRatio'] = np.where(
    flightsDB_new['TotalFlights'] > 0,
    flightsDB_new['TotalFlightsWithCompanions'] / flightsDB_new['TotalFlights'],
    0
)

: 

: 

<a class="anchor" id="7_1_3">

### **7.1.3** Recency
[Back to TOC](#toc)
</a>

In [ ]:
# Latest date in dataset 
latest_date = flightsDB['YearMonthDate'].max()

## Recency (months since last active flight)
# Compute last active flight per customer
active_flights = flightsDB[flightsDB['NumFlights'] > 0]
last_active_flight = active_flights.groupby('Loyalty#')['YearMonthDate'].max()

latest_date = flightsDB['YearMonthDate'].max()
flightsDB_new['Recency'] = (
    (latest_date - flightsDB_new['Loyalty#'].map(last_active_flight)).dt.days / 30
).fillna(99).astype(int) # Fill NaN (no active flights) with 99 to highlight inactivity

: 

: 

<a class="anchor" id="7_1_4">

### **7.1.4** Total Distance per Flight
[Back to TOC](#toc)
</a>

In [ ]:
# Total Distance per Flight
flightsDB_new['DistancePerFlight'] = np.where(
    flightsDB_new['TotalFlights'] > 0,
    flightsDB_new['TotalDistanceKM'] / flightsDB_new['TotalFlights'],
    0
)

: 

: 

<a class="anchor" id="7_2">

## **7.2.** CustomerDB

[Back to TOC](#toc)
</a>

In [ ]:
# Create expanded dataset
customerDB_expanded = customerDB.copy()
customerDB_new = pd.DataFrame()

# Convert enrollment and cancellation dates to datetime
customerDB['EnrollmentDateOpening'] = pd.to_datetime(customerDB_expanded['EnrollmentDateOpening'], errors='coerce')

: 

: 

<a class="anchor" id="7_2_1">

### **7.2.1** IncomeBins
[Back to TOC](#toc)
</a>

In [ ]:
# Compute quartiles
try:
    temp_bins = pd.qcut(income_nonan, q=4, duplicates='drop')
    bins_labels = ['Very Low', 'Low', 'Medium', 'High'][:len(temp_bins.cat.categories)]
    customerDB_expanded['IncomeBins'] = pd.qcut(
        income_nonan,
        q=4,
        labels=bins_labels,
        duplicates='drop'
    )
    # Replace the -1 bin with 'Unknown'
    customerDB_expanded['IncomeBins'] = customerDB_expanded['IncomeBins'].cat.add_categories(['Unknown'])
    customerDB_expanded.loc[income_nonan == -1, 'IncomeBins'] = 'Unknown'
except ValueError:
    customerDB_expanded['IncomeBins'] = pd.Series(['Unknown']*len(customerDB_expanded))

: 

: 

<a class="anchor" id="7_2_2">

### **7.2.2** HouseholdType
[Back to TOC](#toc)
</a>

In [ ]:
# HouseholdType → Marital + Location
customerDB_expanded['HouseholdType'] = (
    customerDB_expanded['Location Code'].fillna('Unknown') + '-' +
    customerDB_expanded['Marital Status'].fillna('Unknown')
)

: 

: 

<a class="anchor" id="7_2_3">

### **7.2.4** MonthsSinceEnrollment
[Back to TOC](#toc)
</a>

In [ ]:
# MonthsSinceEnrollment
reference_date = pd.Timestamp.today()
customerDB_expanded['MonthsSinceEnrollment'] = (
    (reference_date - customerDB['EnrollmentDateOpening']).dt.days / 30
).fillna(0).clip(lower=0)

: 

: 

<a class="anchor" id="7_2_4">

### **7.2.4** CLVBins
[Back to TOC](#toc)
</a>

In [ ]:
## CLV BINS
# Compute quartiles
try:
    temp_bins = pd.qcut(clv_nonan, q=5, duplicates='drop')
    clv_labels = ["Very Low", "Low", "Medium", "High", "Very High"][:len(temp_bins.cat.categories)]
    customerDB_expanded['CLVBins'] = pd.qcut(
        clv_nonan,
        q=5,
        labels=clv_labels,
        duplicates='drop'
    )
    # Replace the -1 bin with 'Unknown'
    customerDB_expanded['CLVBins'] = customerDB_expanded['CLVBins'].cat.add_categories(['Unknown'])
    customerDB_expanded.loc[clv_nonan == -1, 'CLVBins'] = 'Unknown'
except ValueError:
    customerDB_expanded['CLVBins'] = pd.Series(['Unknown']*len(customerDB_expanded))

: 

: 

<a class="anchor" id="7_2_5">

### **7.2.5** CLVvsCityMean
[Back to TOC](#toc)
</a>

In [ ]:
# ---  CLV vs City Mean ---
customerDB_expanded['CLVvsCityMean'] = (
    customerDB_expanded['Customer Lifetime Value'] - customerDB_expanded.groupby('City')['Customer Lifetime Value'] \
    .transform('mean')
)

: 

: 

<a class="anchor" id="7_2_6">

### **7.2.6** Is Active
[Back to TOC](#toc)
</a>

In [ ]:
# IsActive
customerDB_expanded['IsActive'] = customerDB_expanded['CancellationDate'].isna().astype(int)

: 

: 

<a class="anchor" id="7_3">

## **7.3.** Feature Engineering with Merged Datasets

[Back to TOC](#toc)
</a>

<a class="anchor" id="7_3_1">

### **7.3.1** Merge Datasets
[Back to TOC](#toc)
</a>


In [ ]:
# Create a second expanded flightsDB, keeping only one row per customer with aggregated features
flightsDB_expanded = flightsDB.merge(flightsDB_new, on='Loyalty#', how='right')

# Merge customerDB_expanded with flightsDB_expanded on Loyalty#
dfs_merged = customerDB_expanded.join(flightsDB_expanded, on="Loyalty#", how="left")

# Fill only numeric columns with 0
num_cols = dfs_merged.select_dtypes(include=['number']).columns
dfs_merged[num_cols] = dfs_merged[num_cols].fillna(0)

# Keep categorical NA values untouched

: 

: 

<a class="anchor" id="7_3_2">

### **7.3.2** Add Features
[Back to TOC](#toc)
</a>


<a class="anchor" id="7_3_2_1">

### **7.3.2.1** Flights Per Year Of Membership
[Back to TOC](#toc)
</a>


In [ ]:
# Flights per year of membership
dfs_merged["FlightsPerYearOfMembership"] = safe_div(
    dfs_merged["TotalFlights"],
    dfs_merged["MonthsSinceEnrollment"] / 12
)

: 

: 

<a class="anchor" id="7_3_2_2">

### **7.3.2.2** Redemption Rate By Loyalty Status
[Back to TOC](#toc)
</a>

In [ ]:
# Redemption rate by loyalty status
loyalty_points_mean = dfs_merged.groupby("LoyaltyStatus")["TotalPointsRedeemed"].transform("mean")
dfs_merged["RedemptionRateByLoyaltyStatus"] = safe_div(
    dfs_merged["TotalPointsRedeemed"],
    loyalty_points_mean
)

: 

: 

<a class="anchor" id="7_4">

## **7.4.** Summary of Engineered Features

[Back to TOC](#toc)
</a>

To summarise, we chose to keep:

FLIGHTSDB

- `PointsRedemptionRatio`
- `TotalFlightWithCompanionRatio`
- `Recency`
- `DistancePerFlight`

CUSTOMERSDB

- `IncomeBins`
- `HouseholdType`
- `MonthsSinceEnrollment`
- `CLVBins`
- `CLVvsCityMean`
- `IsActive`

MERGED DATASETS

- `FlightsPerYearOfMembership`
- `RedemptionRateByLoyaltyStatus`